# Figure: environmental conditions and greenhouse gas fluxes

Time series of precipitation, soil water content, soil and air temperature
together with the gap-filled NEE, N&#8322;O and CH&#8324; fluxes of the two
treatment parcels over the measurement period.

**Input** `../data/raw/fluxes_meteo_mgmt.parquet`
**Output** `../figures/env+NEE+N2O_*.png`, `../figures/env+CH4_*.png`


# IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd # pandas allows to do a lot of basic data manipulation such as loading csv data files
import matplotlib.pyplot as plt # matplotlib is a basic ploting library of Python
import matplotlib.dates as mdates
import matplotlib.cm as cm
from brokenaxes import brokenaxes
from diive.core.io.files import load_parquet

# LOAD DATA

## Fluxes and meteo

In [ ]:
# Load data
filename = "../data/raw/fluxes_meteo_mgmt.parquet"
data = load_parquet(filename)

# Set time interval
start_date = '2023-10-18'
end_date = '2025-06-05'
data = data[start_date:end_date].copy()

# Define columns of interest
meteo = [
    'prec',
    'swc_0.05', 'swc_0.15', 'swc_0.3',
    'ts_0.05', 'ts_0.15', 'ts_0.3', 'ta',
]
obs_fluxes = [
    'FN2O_L3.3_CUT_50_QCF0_parcelA', 'FN2O_L3.3_CUT_50_QCF0_parcelB',
    'NEE_L3.3_CUT_50_QCF0', 'FCH4_L3.3_CUT_50_QCF0'
]
gf_fluxes = [
    'FN2O_L3.3_CUT_50_QCF_parcelA_gfXGBoost', 'FN2O_L3.3_CUT_50_QCF_parcelB_gfXGBoost',
    'NEE_L3.3_CUT_50_QCF_footprint_gfXGBoost', 'FCH4_L3.3_CUT_50_QCF_footprint_gfXGBoost'
]
gf_flags = [f'FLAG_{c}_ISFILLED' for c in gf_fluxes]

incols = meteo + obs_fluxes + gf_fluxes + gf_flags
data = data[incols].copy()

# Create a df with filtered out days without any observed data
data_filtered = data.copy()
MIN_OBS_PER_DAY = 1 
for col in gf_fluxes:
    flag_col = f'FLAG_{col}_ISFILLED'
    flag = data_filtered[flag_col]
    day_index = data_filtered.index.floor('D')
    obs_count_by_day = (flag == 0).groupby(day_index).sum()
    keep_day = obs_count_by_day >= MIN_OBS_PER_DAY
    keep_rows = day_index.map(keep_day).fillna(False)
    data_filtered.loc[~keep_rows, col] = np.nan

# Calculate daily means and standard deviations
data_daily = pd.DataFrame()
data_daily_filtered = pd.DataFrame()
for col in incols:
    if col == 'prec':
        data_daily[col] = data[col].resample('D').sum()
        data_daily_filtered[col] = data_filtered[col].resample('D').sum()
    else:
        data_daily[col] = data[col].resample('D').mean()
        data_daily[col + '_std'] = data[col].resample('D').std()
        data_daily_filtered[col] = data_filtered[col].resample('D').mean()
        data_daily_filtered[col + '_std'] = data_filtered[col].resample('D').std()

# Plot fluxes
data_daily[gf_fluxes].plot(x_compat=True, subplots=True, figsize=(12, 2*len(gf_fluxes)))
plt.show()

data_daily


# PLOTS

### CONFIGS

In [ ]:
from plot_config import EVENT_LINES, LABELS, COLORS, PERIODS, add_event_lines

# Define cols to access later
N2O_A_col = next((c for c in gf_fluxes if c.startswith("FN2O") and "parcelA" in c), None)
N2O_B_col = next((c for c in gf_fluxes if c.startswith("FN2O") and "parcelB" in c), None)
NEE_col   = next((c for c in gf_fluxes if c.startswith("NEE")), None)
CH4_col = next((c for c in gf_fluxes if c.startswith("FCH4")), None)

## NEE and N2O

In [ ]:
# Function to create period-specific plots with env data + NEE and N2O fluxes
def plot_period(df_data, period_name, start_extent, end_extent, letterx=0.001, lettery=0.99):

    # Filter data for the period
    start_date, end_date = PERIODS[period_name]
    print(f'Plotting period: {period_name} ({start_date} to {end_date})')
    period_data = df_data[start_date:end_date].copy()

    # Transform extent dates to timestamps for later use in event line plotting
    start_ts = pd.to_datetime(start_extent)
    end_ts = pd.to_datetime(end_extent)
    
    # Create subplots
    fig, axs = plt.subplots(4, 1, sharex=True, figsize=(14, 10))
    fig.subplots_adjust(hspace=0.1)

    # Remove the top spine
    for ax in axs:
        ax.spines['top'].set_visible(False)
    
    ########################################################
    # Precipitation and Air Temperature
    ax = axs[0]
    ax.bar(period_data.index, period_data['prec'], color="#1B3A6B", alpha=0.7)
    ax.set_ylabel('Precipitation\n(mm d$^{-1}$)')

    # Create secondary axis for temperature
    ax2 = ax.twinx()
    mean = period_data['ta']
    err = period_data['ta_std']
    # Plot mean and error for Tair on secondary axis
    ax2.plot(period_data.index, mean, color="#B22222", linewidth=2)  
    ax2.fill_between(period_data.index, mean - err, mean + err, color="#B22222", alpha=0.2) 
    ax2.set_ylabel('Ta (°C)')
    ax2.spines['top'].set_visible(False) # remove top spine from secondary axis as well

    ax.text(letterx, lettery, '(a)', transform=ax.transAxes, fontweight='bold', verticalalignment='top')
    ax.set_xlim(start_ts, end_ts)
    
    #######################################################
    # SWC
    ax = axs[1]
    incols = [c for c in period_data.columns if 'swc' in c and '_std' not in c]
    # Convert SWC to WFPS if needed, or just plot SWC directly
    palette = cm.Blues(np.linspace(0.4, 1, len(incols)))
    for i, col in enumerate(incols):
        y = period_data[col]
        label = col.replace('swc_', '') + ' m'
        ax.plot(period_data.index, y, label=label, color=palette[i])
    ax.set_ylabel('SWC\n(%)')
    ax.text(letterx, lettery, '(b)', transform=ax.transAxes, fontweight='bold', verticalalignment='top')
    legend = ax.legend(loc='lower left', frameon=False)
    ax.set_xlim(start_ts, end_ts)
    # Set ticks y axis
    ax.set_yticks(np.arange(10, 34, 5))

    #######################################################
    # NEE
    ax = axs[2]
    mean = period_data[NEE_col]
    err = period_data[f'{NEE_col}_std']
    # Plot mean and error
    ax.fill_between(period_data.index, mean - err, mean + err, color="#009E73", alpha=0.2)
    ax.plot(period_data.index, mean, color="#009E73")
    ax.set_ylabel('NEE\n($\mu$mol m$^{-2}$ s$^{-1}$)')
    ax.text(letterx, lettery, '(c)', transform=ax.transAxes, fontweight='bold', verticalalignment='top')
    # Add zero line
    ax.axhline(0, color='grey', linestyle='--', lw=1)
    # Set ticks y axis
    ax.set_yticks(np.arange(-30, 11, 10))
    # Set x limits
    ax.set_xlim(start_ts, end_ts)

    #######################################################
    # FN2O
    ma = period_data[N2O_A_col]
    mb = period_data[N2O_B_col]
    sa = period_data[f'{N2O_A_col}_std']
    sb = period_data[f'{N2O_B_col}_std']
    # Replace axs[3] with a brokenaxes in the same GridSpec cell
    outer = axs[3].get_subplotspec()   # remember where axs[3] lived
    axs[3].set_visible(False)   # keep it attached to fig, but invisible
    # Create broken y-axis (choose your ranges)
    threshold_min = 3.5 # where break starts
    threshold_max = 5.5 # where break stops
    # Define the max value for y axis
    y_max, y_min = float(np.nanmax([(ma + sa).max(), (mb + sb).max()])), float(np.nanmin([(ma - sa).min(), (mb - sb).min()]))
    y_max = np.ceil(y_max)  # round up nicely
    y_min = np.floor(y_min)
    bax = brokenaxes(
        subplot_spec=outer,
        ylims=((y_min, threshold_min), (threshold_max, y_max)),
        hspace=0.1, # vertical space between slashes
        d = 0.008 # slashes length
    )
    # Define labels
    label_A, label_B = LABELS['A'], LABELS['B']
    # Plot
    bax.fill_between(period_data.index, ma - sa, ma + sa, alpha=0.2, color=COLORS['A'])
    bax.plot(period_data.index, ma, color=COLORS['A'], label=label_A, alpha=0.7)
    bax.fill_between(period_data.index, mb - sb, mb + sb, alpha=0.2, color=COLORS['B'])
    bax.plot(period_data.index, mb, color=COLORS['B'], label=label_B, alpha=0.7)
    # y label will be set later
    for a in bax.axs:
        a.set_ylabel("")
    # Panel label: brokenaxes has its own axes list; use the *top* one for transAxes
    bax.axs[0].text(
        letterx, lettery, '(d)',
        transform=bax.axs[0].transAxes,
        fontweight='bold', va='top'
    )
    # Add zero line
    bax.axhline(0, color='grey', linestyle='--', lw=1)
    # Legend
    bax.legend(loc='upper left', bbox_to_anchor=(0.02, 1), frameon=False)
    # X limits
    bax.set_xlim(start_ts, end_ts)

    #######################################################

    # Add event lines and labels
    add_event_lines(axs[0], start_ts=start_ts, end_ts=end_ts, mgmt=True, height=1.03, x_spacing_days=5)
    for ax in axs[1:3]:  # add to the middle two axes (SWC and NEE)
        add_event_lines(ax, start_ts=start_ts, end_ts=end_ts, mgmt=False, height=1.01, x_spacing_days=5)
    for a in bax.axs:  # add to all broken axes (N2O)
        add_event_lines(a, start_ts=start_ts, end_ts=end_ts, mgmt=False, height=1.01, x_spacing_days=3)
            
    # Format x-axis (use the bottom internal axis of brokenaxes)
    bottom_ax = bax.axs[-1]
    bottom_ax.xaxis.set_major_locator(mdates.MonthLocator())
    bottom_ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    secax = bottom_ax.secondary_xaxis('bottom')
    secax.xaxis.set_major_locator(mdates.YearLocator())
    secax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    secax.tick_params(axis='x', length=0, pad=30)

    # Add the right spine and ticks to the broken plot
    for a in bax.axs:
        a.spines["right"].set_visible(True)
        a.yaxis.set_ticks_position("both")
        a.tick_params(axis="y", right=True, labelright=False)
    # Add ticks also to right spine for the 2nd and 3rd plot
    for ax in axs[1:3]:
        ax.yaxis.set_ticks_position("both")
        ax.tick_params(axis="y", right=True, labelright=False)

    # set position of y labels
    xlab = -0.05  # move left (more negative = further left)
    for ax in axs:
        ax.yaxis.set_label_coords(xlab, 0.5)
    bax.axs[-1].set_ylabel('N$_2$O flux\n(nmol m$^{-2}$ s$^{-1}$)')
    bax.axs[-1].yaxis.set_label_coords(xlab, 0.8) # somehow with 0.5 it would be off

    return fig

In [ ]:
# Plot
# Choose if using the filtered fluxes or the complete version
df = data_daily_filtered 
#df = data_daily
period = "complete_period"
start = '2023-10-15'
end = '2024-08-21'

plt.rcParams.update({'font.size': 12})  # Set global font size
fig = plot_period(df, period, start_extent=start, end_extent=end, letterx=0.015, lettery=0.99)
fig.align_labels()  # align y labels
plt.savefig(f'../figures/env+NEE+N2O_{period.lower()}.png', dpi=300, bbox_inches='tight')
plt.show()

## CH4

In [ ]:
# Function to create period-specific plots with env data + NEE and N2O fluxes
def plot_methane(df_data, period_name, start_extent, end_extent, letterx=0.001, lettery=0.99):
    
    # Filter data for the period
    start_date, end_date = PERIODS[period_name]
    print(f'Plotting period: {period_name} ({start_date} to {end_date})')
    period_data = df_data[start_date:end_date].copy()

    # Transform extent dates to timestamps for later use in event line plotting
    start_ts = pd.to_datetime(start_extent)
    end_ts = pd.to_datetime(end_extent)
    
    # Create subplots
    fig, axs = plt.subplots(3, 1, sharex=True, figsize=(14, 7.5))
    fig.subplots_adjust(hspace=0.05)

    # Remove the top spine
    for ax in axs:
        ax.spines['top'].set_visible(False)
    
    ########################################################
    # Precipitation and Air Temperature
    ax = axs[0]
    ax.bar(period_data.index, period_data['prec'], color="#1B3A6B", alpha=0.7)
    ax.set_ylabel('Precipitation\n(mm d$^{-1}$)')

    # Create secondary axis for temperature
    ax2 = ax.twinx()
    mean = period_data['ta']
    err = period_data['ta_std']
    # Plot mean and error for Tair on secondary axis
    ax2.plot(period_data.index, mean, color="#B22222", linewidth=2)  
    ax2.fill_between(period_data.index, mean - err, mean + err, color="#B22222", alpha=0.2) 
    ax2.set_ylabel('Ta (°C)')
    ax2.spines['top'].set_visible(False) # remove top spine from secondary axis as well

    ax.text(letterx, lettery, '(a)', transform=ax.transAxes, fontweight='bold', verticalalignment='top')
    ax.set_xlim(start_ts, end_ts)
    
    #######################################################
    # SWC
    ax = axs[1]
    incols = [c for c in period_data.columns if 'swc' in c and '_std' not in c]
    # Convert SWC to WFPS if needed, or just plot SWC directly
    palette = cm.Blues(np.linspace(0.4, 1, len(incols)))
    for i, col in enumerate(incols):
        y = period_data[col]
        label = col.replace('swc_', '') + ' m'
        ax.plot(period_data.index, y, label=label, color=palette[i])
    ax.set_ylabel('SWC\n(%)')
    ax.text(letterx, lettery, '(b)', transform=ax.transAxes, fontweight='bold', verticalalignment='top')
    legend = ax.legend(loc='lower left', frameon=False)
    ax.set_xlim(start_ts, end_ts)
    # Set ticks y axis
    ax.set_yticks(np.arange(10, 34, 5))

    #######################################################
    # FCH4
    ax = axs[2]
    mean = period_data[CH4_col]
    err = period_data[f'{CH4_col}_std']
    # Plot mean and error
    ax.fill_between(period_data.index, mean - err, mean + err, color="#009E73", alpha=0.2)
    ax.plot(period_data.index, mean, color="#009E73")
    ax.set_ylabel('CH$_{4}$\n(nmol m$^{-2}$ s$^{-1}$)')
    ax.text(letterx, lettery, '(c)', transform=ax.transAxes, fontweight='bold', verticalalignment='top')
    # Add zero line
    ax.axhline(0, color='grey', linestyle='--', lw=1)
    # Set ticks y axis
 #   ax.set_yticks(np.arange(-30, 11, 10))
    # Set x limits
    ax.set_xlim(start_ts, end_ts)

    #######################################################
    # Add management events
    add_event_lines(axs[0], start_ts=start_ts, end_ts=end_ts, mgmt=True, height=1.03, x_spacing_days=5)
    for ax in axs[1:]:  # only lines for the 2nd and 3rd plot (SWC and CH4)
        add_event_lines(ax, start_ts=start_ts, end_ts=end_ts, mgmt=False, height=1.01, x_spacing_days=5)
            
    # Add ticks also to right spine 
    for ax in axs:
        ax.yaxis.set_ticks_position("both")
        ax.tick_params(axis="y", right=True, labelright=False)

    return fig

In [ ]:
# Plot
# Choose if using the filtered fluxes or the complete version
df = data_daily_filtered 
#df = data_daily
period = "complete_period"
start = '2023-10-15'
end = '2024-08-21'

plt.rcParams.update({'font.size': 12})  # Set global font size
fig = plot_methane(df, period, start_extent=start, end_extent=end, letterx=0.015, lettery=0.99)
fig.align_labels()  # align y labels
plt.savefig(f'../figures/env+CH4_{period.lower()}.png', dpi=300, bbox_inches='tight')
plt.show()